# RegImpact v0.6B — regulatory-clause classifier

This notebook is an auditable operator interface over the repository scripts. It does not bypass the v0.6A dataset audit, evaluate repeatedly on the test set, or promote a model automatically.

In [ ]:
from pathlib import Path
import json, os, subprocess

API_ROOT = Path.cwd().resolve()
if API_ROOT.name == 'notebooks':
    API_ROOT = API_ROOT.parent / 'apps' / 'api'
DATASET_ID = os.environ.get('REGIMPACT_DATASET_ID', 'clauses-v1')
DATASET = Path(os.environ.get('REGIMPACT_DATASET', '/secure/clauses-v1.jsonl'))
AUDIT = Path(os.environ.get('REGIMPACT_DATASET_AUDIT', '/secure/clauses-v1-audit.json'))
OUTPUT = Path(os.environ.get('REGIMPACT_MODEL_OUTPUT', '/secure/model-registry/regimpact-clause-v1'))
BASELINE_REPORT = OUTPUT.parent / f'{DATASET_ID}-tfidf-baseline.json'
BASE_MODEL_REVISION = os.environ['REGIMPACT_BASE_MODEL_REVISION']
TRAINING_COMMIT = os.environ['REGIMPACT_TRAINING_COMMIT']

## 1. Verify the independent dataset audit

In [ ]:
audit = json.loads(AUDIT.read_text())
assert audit['status'] == 'ready' and audit['failures'] == []
assert audit['dataset_id'] == DATASET_ID
{key: audit[key] for key in ('dataset_id', 'dataset_sha256', 'examples', 'documents', 'regulators', 'labels', 'agreement_rate')}

## 2. Run the TF-IDF + logistic-regression baseline

In [ ]:
subprocess.run([
    'python', str(API_ROOT / 'scripts/evaluate_clause_baseline.py'),
    '--dataset', str(DATASET), '--dataset-id', DATASET_ID,
    '--dataset-audit', str(AUDIT), '--output', str(BASELINE_REPORT),
], cwd=API_ROOT, check=True)
json.loads(BASELINE_REPORT.read_text())

## 3. Fine-tune and evaluate the encoder

Temperature and abstention threshold are selected only on validation data. The isolated test partition is evaluated once when this command completes.

In [ ]:
training = subprocess.run([
    'python', str(API_ROOT / 'scripts/train_clause_classifier.py'),
    '--dataset', str(DATASET), '--dataset-id', DATASET_ID,
    '--dataset-audit', str(AUDIT), '--output', str(OUTPUT),
    '--base-model', 'nlpaueb/legal-bert-base-uncased',
    '--base-model-revision', BASE_MODEL_REVISION,
    '--training-commit', TRAINING_COMMIT,
    '--epochs', '3', '--seed', '42',
], cwd=API_ROOT)
print('exit code:', training.returncode, '(0=qualified, 2=not qualified)')

In [ ]:
manifest = json.loads((OUTPUT / 'manifest.json').read_text())
{key: manifest[key] for key in ('model_id', 'dataset_sha256', 'macro_f1', 'per_class_f1', 'expected_calibration_error', 'coverage', 'covered_accuracy', 'promotion_failures', 'promoted')}

## 4. Human-controlled promotion

Run only after error analysis, slice review, model-card completion, and an authorized approval. Promotion writes a receipt that binds the dataset audit, manifest, model files, training commit, approver, and timestamp.

In [ ]:
# Intentionally disabled by default. Set these only after the review is complete.
APPROVER = os.environ.get('REGIMPACT_MODEL_APPROVER')
APPROVED_AT = os.environ.get('REGIMPACT_MODEL_APPROVED_AT')
TRAINING_COMMIT = os.environ.get('REGIMPACT_TRAINING_COMMIT')
if not all((APPROVER, APPROVED_AT, TRAINING_COMMIT)):
    print('Promotion not requested; review variables are unset.')
else:
    subprocess.run([
        'python', str(API_ROOT / 'scripts/promote_clause_classifier.py'),
        '--artifact', str(OUTPUT), '--dataset-audit', str(AUDIT),
        '--approver', APPROVER, '--approved-at', APPROVED_AT,
        '--training-commit', TRAINING_COMMIT,
    ], cwd=API_ROOT, check=True)